# Data Collection & Merging
## DSA 210 — Betül Merey (33963)

This notebook merges three data sources to build the analysis dataset:
- **EM-DAT**: earthquake events (casualties, damage) since 1990
- **World Bank API**: GDP growth, GDP per capita, unemployment by country/year
- **Output**: `data/processed/merged_dataset.csv`

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/DSA210_PROJ')
print("Working directory:", os.getcwd())

Mounted at /content/drive
Working directory: /content/drive/MyDrive/DSA210_PROJ


In [2]:
import json
import time
import urllib.request
from pathlib import Path

import pandas as pd
import numpy as np
import requests

print("Libraries loaded.")

Libraries loaded.


In [3]:
ROOT = Path(r"c:/Users/erdem/OneDrive/Desktop/btl")
EMDAT_XLSX = ROOT / "emdat_data.xlsx"
USGS_CSV = ROOT / "usgs_1900.csv"
OUT_CSV = ROOT / "merged_dataset.csv"

INDICATORS = {
    "NY.GDP.MKTP.KD.ZG": "gdp_growth",        # GDP growth (annual %)
    "NY.GDP.PCAP.CD":    "gdp_per_capita",    # GDP per capita (current US$)
    "SL.UEM.TOTL.ZS":    "unemployment",      # Unemployment rate (%)
}

INCOME_THRESHOLDS = [   # World Bank 2024
    (1085,  "Low"),
    (4255,  "Lower-Middle"),
    (13205, "Upper-Middle"),
    (float("inf"), "High"),
]

## Step 1 — Load EM-DAT Earthquake Data

EM-DAT (Emergency Events Database) records disaster events worldwide.
We filter for earthquakes starting from 1990 and keep only the columns relevant to our analysis.

In [4]:
emdat_raw = pd.read_excel('data/raw/emdat/emdat_data.xlsx')

# Filter: earthquakes only, 1990 onwards
emdat = emdat_raw[emdat_raw['Disaster Type'] == 'Earthquake'].copy()
emdat = emdat[emdat['Start Year'] >= 1990].copy()

# Keep and rename relevant columns
keep = {
    'ISO'                              : 'iso3',
    'Country'                          : 'country',
    'Start Year'                       : 'year',
    'Magnitude'                        : 'magnitude',
    'Total Deaths'                     : 'total_deaths',
    'Total Affected'                   : 'total_affected',
    "Total Damage ('000 US$)"          : 'total_damage_kUSD',
    "Total Damage, Adjusted ('000 US$)": 'total_damage_adj_kUSD',
}
emdat = emdat[list(keep.keys())].rename(columns=keep)
emdat = emdat.dropna(subset=['iso3', 'year']).reset_index(drop=True)
emdat['year'] = emdat['year'].astype(int)

print(f"Earthquakes loaded: {len(emdat)} events")
print(f"Countries: {emdat['iso3'].nunique()}")
print(f"Year range: {emdat['year'].min()} – {emdat['year'].max()}")
emdat.head()

Earthquakes loaded: 959 events
Countries: 110
Year range: 1990 – 2026


,iso3,country,year,magnitude,total_deaths,total_affected,total_damage_kUSD,total_damage_adj_kUSD
0,PHL,Philippines,1990,6.6,1.0,34504.0,900.0,2161.0
1,IDN,Indonesia,1990,7.8,5.0,7036.0,NaN,NaN
2,BGR,Bulgaria,1990,7.0,1.0,NaN,NaN,NaN
3,ROU,Romania,1990,7.0,9.0,700.0,NaN,NaN
4,SUN,Soviet Union,1990,7.0,4.0,NaN,NaN,NaN


## Step 2 — Fetch World Bank Indicators

We pull three macroeconomic indicators for all countries (1985–2030) via the World Bank API:
- **GDP growth** (annual %) — to measure economic impact
- **GDP per capita** (USD) — to classify income groups
- **Unemployment rate** (%) — additional economic context

In [5]:
def fetch_wb(indicator_code):
    url = (
        f"https://api.worldbank.org/v2/country/all/indicator/{indicator_code}"
        f"?format=json&per_page=20000&date=1985:2030"
    )

    for attempt in range(3):  # try up to 3 times in case of network issues because it could not finish in time
        try:
            # send a GET request to the World Bank API
            # timeout=120 means: if no response in 120 seconds, give up
            resp = requests.get(url, timeout=120)

            # payload is a list: payload[0] = metadata, payload[1] = actual data
            payload = resp.json()

            # loop through each record and extract only the fields we need
            # skip records where country code or value is missing
            records = [
                {'iso3': r['countryiso3code'], 'year': int(r['date']), 'value': r['value']}
                for r in payload[1]
                if r['countryiso3code'] and r['value'] is not None
            ]

            # convert list of records to a DataFrame and return it
            return pd.DataFrame(records)

        except Exception as e:
            import time
            # wait 5 seconds before trying again
            # this gives the server time to recover
            time.sleep(5)

    # if all 3 attempts fail, return an empty DataFrame so the code doesn't crash
    return pd.DataFrame(columns=['iso3', 'year', 'value'])

# Fetch all three indicators
print("Fetching GDP growth...")
gdp_df   = fetch_wb('NY.GDP.MKTP.KD.ZG')
print(f"  {len(gdp_df)} rows")

print("Fetching GDP per capita...")
gdppc_df = fetch_wb('NY.GDP.PCAP.CD')
print(f"  {len(gdppc_df)} rows")

print("Fetching unemployment...")
unem_df  = fetch_wb('SL.UEM.TOTL.ZS')
print(f"  {len(unem_df)} rows")

print("Done.")



Fetching GDP growth...
  9705 rows
Fetching GDP per capita...
  9818 rows
Fetching unemployment...
  8071 rows
Done.


In [6]:
# Pivot each indicator: rows = country, columns = year
gdp_pivot   = gdp_df.pivot(index='iso3', columns='year', values='value')
gdppc_pivot = gdppc_df.pivot(index='iso3', columns='year', values='value')
unem_pivot  = unem_df.pivot(index='iso3', columns='year', values='value')

print("GDP growth pivot shape  :", gdp_pivot.shape)
print("GDP per capita pivot shape:", gdppc_pivot.shape)
print("Unemployment pivot shape:", unem_pivot.shape)

GDP growth pivot shape  : (258, 40)
GDP per capita pivot shape: (258, 40)
Unemployment pivot shape: (231, 35)


## Step 3 — Compute Recovery Metrics

For each earthquake event we calculate:
- **Pre-event GDP baseline**: average GDP growth in the 3 years before the earthquake
- **GDP drop**: difference between event-year GDP and the baseline
- **Recovery time**: how many years until GDP growth returned to baseline level (max 5 years)

In [7]:
def get_value(pivot, iso3, year):
    if pivot.empty or iso3 not in pivot.index or year not in pivot.columns:
        return np.nan
    return pivot.loc[iso3, year]

def compute_recovery(row):
    iso3 = row['iso3']
    year = row['year']

    # 3-year pre-event baseline
    baseline = [get_value(gdp_pivot, iso3, y) for y in range(year - 3, year)]
    baseline = [v for v in baseline if not np.isnan(v)]
    pre_avg  = np.mean(baseline) if baseline else np.nan

    gdp_event = get_value(gdp_pivot, iso3, year)
    gdp_drop  = (gdp_event - pre_avg) if not (np.isnan(gdp_event) or np.isnan(pre_avg)) else np.nan

    # Recovery: first year post-event where GDP >= baseline
    recovery_years = np.nan
    for lag in range(1, 6):
        val = get_value(gdp_pivot, iso3, year + lag)
        if not np.isnan(val) and not np.isnan(pre_avg) and val >= pre_avg:
            recovery_years = lag
            break

    return pd.Series({
        'pre_event_avg_gdp': round(pre_avg, 4)    if not np.isnan(pre_avg)    else np.nan,
        'gdp_event_year'   : round(gdp_event, 4)  if not np.isnan(gdp_event)  else np.nan,
        'gdp_drop'         : round(gdp_drop, 4)   if not np.isnan(gdp_drop)   else np.nan,
        'recovery_years'   : recovery_years,
    })

recovery = emdat.apply(compute_recovery, axis=1).reset_index(drop=True)
print("Recovery metrics computed.")
recovery.head()

Recovery metrics computed.


,pre_event_avg_gdp,gdp_event_year,gdp_drop,recovery_years
0,5.7476,3.0827,-2.6649,NaN
1,6.0543,7.2421,1.1878,1.0
2,4.5698,-9.1174,-13.6872,NaN
3,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN


## Step 4 — Merge Everything & Assign Income Groups

We combine EM-DAT events with World Bank indicators, then classify each country into an income group based on GDP per capita (World Bank 2024 thresholds).

In [8]:
merged = pd.concat([emdat, recovery], axis=1)

# Add GDP per capita and unemployment for event year
merged['gdp_per_capita_event_year'] = merged.apply(
    lambda r: get_value(gdppc_pivot, r['iso3'], r['year']), axis=1
)
merged['unemployment_event_year'] = merged.apply(
    lambda r: get_value(unem_pivot, r['iso3'], r['year']), axis=1
)

# Assign income group based on GDP per capita
INCOME_BINS   = [0, 1_085, 4_255, 13_205, float('inf')]
INCOME_LABELS = ['Low', 'Lower-Middle', 'Upper-Middle', 'High']

merged['income_group'] = pd.cut(
    merged['gdp_per_capita_event_year'],
    bins=INCOME_BINS,
    labels=INCOME_LABELS
)

print(f"Final dataset: {len(merged)} rows × {len(merged.columns)} columns")
print(f"\nIncome group distribution:")
print(merged['income_group'].value_counts())
merged.head()

Final dataset: 959 rows × 15 columns

Income group distribution:
income_group
Lower-Middle    327
Upper-Middle    233
Low             207
High            122
Name: count, dtype: int64


,iso3,country,year,magnitude,total_deaths,total_affected,total_damage_kUSD,total_damage_adj_kUSD,pre_event_avg_gdp,gdp_event_year,gdp_drop,recovery_years,gdp_per_capita_event_year,unemployment_event_year,income_group
0,PHL,Philippines,1990,6.6,1.0,34504.0,900.0,2161.0,5.7476,3.0827,-2.6649,NaN,803.572588,NaN,Low
1,IDN,Indonesia,1990,7.8,5.0,7036.0,NaN,NaN,6.0543,7.2421,1.1878,1.0,578.420121,NaN,Low
2,BGR,Bulgaria,1990,7.0,1.0,NaN,NaN,NaN,4.5698,-9.1174,-13.6872,NaN,2366.529821,NaN,Lower-Middle
3,ROU,Romania,1990,7.0,9.0,700.0,NaN,NaN,NaN,NaN,NaN,NaN,1648.485230,NaN,Lower-Middle
4,SUN,Soviet Union,1990,7.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6. Enrich Dataset — USGS Depth & Tsunami Flag

In [19]:
# Match by country ISO + year instead of coordinates
usgs = pd.read_csv('data/raw/usgs/usgs_1900.csv',
                   usecols=['time','latitude','longitude','mag','depth'])
usgs['time'] = pd.to_datetime(usgs['time'], errors='coerce', utc=True)
usgs = usgs.dropna(subset=['time','latitude','longitude','mag'])
usgs['year'] = usgs['time'].dt.year

# For each event use magnitude to find the closest USGS match in same year
depths = []
for _, row in merged.iterrows():
    cand = usgs[usgs['year'] == row['year']]
    if cand.empty or pd.isna(row['magnitude']):
        depths.append(None)
        continue
    # Match by closest magnitude in same year
    diff = (cand['mag'] - row['magnitude']).abs()
    idx = diff.idxmin()
    if diff.loc[idx] > 0.5:  # If the magnitude difference is greater than 0.5, no match
        depths.append(None)
    else:
        depths.append(float(cand.loc[idx, 'depth'])
                      if pd.notna(cand.loc[idx, 'depth']) else None)

merged['usgs_depth_km'] = depths
merged['is_tsunami'] = 0

merged.to_csv('data/processed/merged_dataset.csv', index=False)
print(f"Saved: {len(merged)} rows x {len(merged.columns)} cols")
print(f"usgs_depth_km dolu: {merged['usgs_depth_km'].notna().sum()} / {len(merged)}")

Saved: 959 rows x 17 cols
usgs_depth_km dolu: 751 / 959


## Step 5 — Save Merged Dataset

In [20]:
merged.to_csv('data/processed/merged_dataset.csv', index=False)
print("Saved → data/processed/merged_dataset.csv")

# Quick check
df_check = pd.read_csv('data/processed/merged_dataset.csv')
print(f"Verified: {df_check.shape[0]} rows, {df_check.shape[1]} columns")

Saved → data/processed/merged_dataset.csv
Verified: 959 rows, 17 columns


---

## AI Usage Statement

AI assistance (Claude by Anthropic) was used in this notebook in the following ways:

- **Bug fixing**: AI identified the root cause of the 959 phantom row bug — a pandas index misalignment between `emdat` (index 4098, 4104...) and `recovery` (index 0, 1, 2...) during `pd.concat`. The fix was adding `.reset_index(drop=True)` to both DataFrames before concatenation.
- **API structure**: AI helped structure the World Bank API fetch function with retry logic and timeout handling.
- **Recovery metric design**: AI clarified the 3-year pre-event baseline approach and the 5-year cap for recovery time calculation.
- **Code suggestions**: Syntax help for `pivot_table`, `pd.cut` for income group binning, and `pd.to_numeric` for safe type conversion.

The decision to use EM-DAT as the primary data source, the choice of World Bank indicators, income group thresholds (World Bank 2024), and the overall merging logic were decided by the author.